# Pattern: Sliding Window

## The core insight

Sliding window is the answer to problems about **contiguous subarrays or substrings**. The brute force is to check every possible window — O(n²) or worse. Sliding window keeps a window of elements and moves it across the array, adding one element on the right and removing one on the left. Each element enters and exits the window exactly once — O(n) total.

## How to recognise this pattern

Look for these signals in the problem:
- "Contiguous subarray" or "substring"
- "Longest / shortest" subarray satisfying some condition
- "Maximum / minimum sum" of a subarray of size k
- "Contains all characters" or "no repeating characters"

## The two variants

**Fixed-size window** — the window size `k` is given. Slide it forward: add `arr[right]`, remove `arr[right - k]`.

```python
window_sum = sum(arr[:k])       # initialise first window
for i in range(k, len(arr)):
    window_sum += arr[i] - arr[i - k]   # slide: add new, remove old
```

**Variable-size window** — you expand the right edge until the window becomes invalid, then shrink the left edge until it's valid again.

```python
left = 0
for right in range(len(arr)):
    # expand: add arr[right] to window
    
    while window is invalid:
        # shrink: remove arr[left] from window
        left += 1
    
    # window is now valid — update result
    result = max(result, right - left + 1)
```

**Window size formula:** `right - left + 1`

---

## Problem 1 — Best Time to Buy and Sell Stock (LC #121) ⭐ Easy

Given an array of daily prices, find the maximum profit from one buy and one sell. You must buy before you sell.

```
[7, 1, 5, 3, 6, 4]  →  5   (buy at 1, sell at 6)
[7, 6, 4, 3, 1]     →  0   (prices only fall — don't trade)
[1, 2]              →  1
```

**Think before coding:**

This is a sliding window in disguise. Think of `left` as the buy day and `right` as the sell day. At each new day (`right`), you want to know: is there a better buy price to the left of me?

- What do you track as you scan through prices?
- When do you update your best buy price?
- When do you update your best profit?
- What do you return if prices only fall?

In [ ]:
# Trace [7, 1, 5, 3, 6, 4] step by step before writing code:
# Day 0: price=7, min_price=?, max_profit=?
# Day 1: price=1, min_price=?, max_profit=?
# Day 2: price=5, profit if sold today=?, max_profit=?
# Day 3: price=3, profit if sold today=?, max_profit=?
# Day 4: price=6, profit if sold today=?, max_profit=?
# Day 5: price=4, profit if sold today=?, max_profit=?

In [ ]:
from typing import List

def maxProfit(prices: List[int]) -> int:
    pass

assert maxProfit([7, 1, 5, 3, 6, 4]) == 5
assert maxProfit([7, 6, 4, 3, 1]) == 0
assert maxProfit([1, 2]) == 1
print('Problem 1: passed')

<details>
<summary>💡 Hint — what to initialise and update (click to expand)</summary>

Track two values: `min_price = float('inf')` and `max_profit = 0`. For each price:
- If it's lower than `min_price`, update `min_price` (new best buy day)
- Otherwise, check if `price - min_price` beats `max_profit`

These two cases are mutually exclusive — you either found a cheaper buy, or you evaluate today as a potential sell day.
</details>

In [ ]:
# ============================================================
# SOLUTION — only run after a genuine attempt
# ============================================================

def maxProfit_solution(prices: List[int]) -> int:
    min_price = float('inf')
    max_profit = 0
    for price in prices:
        if price < min_price:
            min_price = price
        elif price - min_price > max_profit:
            max_profit = price - min_price
    return max_profit

---
## Problem 2 — Longest Substring Without Repeating Characters (LC #3) ⭐⭐ Easy/Medium

Find the length of the longest substring containing no duplicate characters.

```
'abcabcbb'  →  3   ('abc')
'bbbbb'     →  1   ('b')
'pwwkew'    →  3   ('wke')
''          →  0
```

**This is the canonical variable-size sliding window problem.**

The window `[left, right]` represents the current substring. You want to keep it valid (no duplicates). As you expand right, if a duplicate appears, you must shrink from the left until it's gone.

**Before coding:**
- What data structure tracks what's in the current window?
- What makes the window invalid?
- How do you shrink the window — what do you remove from your tracking structure when `left` advances?
- When do you update the result?

In [ ]:
# Trace 'abcabcbb' step by step:
# right=0 (a): window={a}, valid, length=1
# right=1 (b): window={a,b}, valid, length=2
# right=2 (c): window={a,b,c}, valid, length=3
# right=3 (a): 'a' is in window — shrink left until 'a' is removed. What happens?
# right=4 (b): 'b' is in window — shrink left. What happens?
# ...

# Data structure to track window contents: set or dict? Why?
# (Hint: do you need to know WHERE in the window each character is, or just WHETHER it's there?)

In [ ]:
def lengthOfLongestSubstring(s: str) -> int:
    pass

assert lengthOfLongestSubstring('abcabcbb') == 3
assert lengthOfLongestSubstring('bbbbb') == 1
assert lengthOfLongestSubstring('pwwkew') == 3
assert lengthOfLongestSubstring('') == 0
print('Problem 2: passed')

<details>
<summary>💡 Hint — the shrinking step (click to expand)</summary>

Use a `seen = set()` to track characters in the current window. When `s[right]` is already in `seen`, you must remove characters from the left:
```python
while s[right] in seen:
    seen.discard(s[left])
    left += 1
```
After the while loop, add `s[right]` to `seen` and update the result with `right - left + 1`.
</details>

In [ ]:
# ============================================================
# SOLUTION
# ============================================================

def lengthOfLongestSubstring_solution(s: str) -> int:
    seen = set()
    left = 0
    result = 0
    for right in range(len(s)):
        while s[right] in seen:
            seen.discard(s[left])
            left += 1
        seen.add(s[right])
        result = max(result, right - left + 1)
    return result

---
## Problem 3 — Maximum Subarray (LC #53) ⭐ Medium — Kadane's Algorithm

Find the contiguous subarray with the largest sum.

```
[-2, 1, -3, 4, -1, 2, 1, -5, 4]  →  6   (subarray [4,-1,2,1])
[1]                               →  1
[-1, -2, -3]                      →  -1
```

**The key decision at each element:**

At each position, you have two choices:
1. Extend the current subarray by including this element
2. Start a fresh subarray from this element

Which do you choose? The one that gives a larger value. In code: `current_sum = max(num, current_sum + num)`. If `current_sum + num` is less than `num` alone, the current subarray is dragging you down — cut it and start fresh.

**Before coding:** trace `[-2, 1, -3, 4, -1, 2, 1, -5, 4]` step by step. At each element, write out `current_sum` and `max_sum`.

In [ ]:
# Trace by hand:
# num=-2: extend or restart? current_sum=?, max_sum=?
# num=1:  extend or restart? current_sum=?, max_sum=?
# num=-3: extend or restart? current_sum=?, max_sum=?
# num=4:  extend or restart? current_sum=?, max_sum=?
# num=-1: extend or restart? current_sum=?, max_sum=?
# num=2:  extend or restart? current_sum=?, max_sum=?
# num=1:  extend or restart? current_sum=?, max_sum=?
# num=-5: extend or restart? current_sum=?, max_sum=?
# num=4:  extend or restart? current_sum=?, max_sum=?

In [ ]:
def maxSubArray(nums: List[int]) -> int:
    pass

assert maxSubArray([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6
assert maxSubArray([1]) == 1
assert maxSubArray([-1, -2, -3]) == -1
print('Problem 3: passed')

In [ ]:
# ============================================================
# SOLUTION
# ============================================================

def maxSubArray_solution(nums: List[int]) -> int:
    current_sum = nums[0]
    max_sum = nums[0]
    for num in nums[1:]:
        current_sum = max(num, current_sum + num)  # extend or restart
        max_sum = max(max_sum, current_sum)
    return max_sum

---
## After completing all problems — cold rewrite

Close the notebook. Write `maxProfit` and `lengthOfLongestSubstring` from scratch in a blank file.

**The question to ask on a new problem:**
> *"Does the problem ask about a contiguous range of elements? Can I represent the answer as maintaining a window and sliding it across?"*

**Fixed window:** size k is fixed, slide forward.  
**Variable window:** expand right, shrink left when invalid, track the best valid window seen.